<a href="https://colab.research.google.com/github/taeyoung0524/LoRA-VLM/blob/main/2_PEFT_%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2주차 통합 실습: PEFT와 LoRA 구조-비용 분석

## 주차 목표
LoRA의 low-rank update 구조를 직접 구현하고, full fine-tuning 대비 trainable parameter와 parameter-state memory 비용을 수치로 비교한다.

## Section 구성
- **Section 1:** `nn.Linear` 하나에 LoRA adapter를 직접 붙이고 full fine-tuning과 rank별 LoRA 학습 결과를 비교한다.
- **Section 2:** SmolVLM 기준 full fine-tuning과 reference rank LoRA의 trainable parameter, parameter-state memory 비용을 비교한다.

COCO image captioning LoRA fine-tuning 실습은 `2.5-PEFT.ipynb`에서 이어서 실행한다.

## Colab 환경 설정

Colab에서 실행할 때 Google Drive를 mount하고 프로젝트 경로를 import path에 추가한다. Drive 폴더명이 다르면 `DRIVE_PROJECT`만 바꾼다.


In [ ]:
# Colab 환경 설정
import importlib.util
import sys

# Drive 경로 지정 - Google Drive가 Colab에 연결되면 /content/drive/MyDrive 경로로 접근할 수 있다.
DRIVE_PROJECT = '/content/drive/MyDrive'

# Colab 환경인지 확인 후 Drive 연결
if importlib.util.find_spec('google.colab') is not None:
    from google.colab import drive

    drive.mount('/content/drive')
    if DRIVE_PROJECT not in sys.path:
        sys.path.insert(0, DRIVE_PROJECT)

# Drive I/O가 느릴 때만 사용한다.
# gd_mount: Colab Drive 프로젝트를 로컬 작업 경로로 복사하고 import path를 맞춥니다.
# from utils.gd_mount import setup_colab_workdir
# setup_colab_workdir(drive_project=DRIVE_PROJECT, mount_drive=False)

## 로컬 환경 설정

로컬 Jupyter나 VS Code에서 실행할 때 현재 경로의 부모를 탐색해 저장소 루트를 찾는다. 로컬 dependency는 notebook 설치 셀보다 `uv sync`로 맞춘 환경을 우선한다.


In [ ]:
# 로컬 환경 설정
# 로컬 노트북/터미널에서 실행할 때 현재 위치의 부모 폴더를 거슬러 올라가며 프로젝트 루트를 찾는다.
# 프로젝트 루트에는 index.md가 있으므로, 어느 하위 폴더에서 열어도 utils import가 안정적으로 동작한다.
import os
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / 'index.md').exists():
        LOCAL_PROJECT_PATH = candidate
        break
else:
    # index.md를 찾지 못하면 현재 작업 폴더를 그대로 사용한다.
    LOCAL_PROJECT_PATH = Path.cwd()

# 노트북의 상대 경로 기준을 프로젝트 루트로 맞춘다.
os.chdir(LOCAL_PROJECT_PATH)
local_project_path = str(LOCAL_PROJECT_PATH)
if local_project_path not in sys.path:
    sys.path.insert(0, local_project_path)
print(f'Local project root: {LOCAL_PROJECT_PATH}')


## 필요 라이브러리 설치

2주차 notebook에서 직접 사용하는 패키지만 설치한다. 1주차와 동일하게 Colab 기본 `torch`/`torchvision`/`torchaudio`와 NumPy/Pandas는 보존하고, 로컬에서는 이 셀보다 `uv sync`로 맞춘 저장소 환경을 우선한다.


Colab에는 기본적으로 설치된 것들이 있지만, LoRA 실습에 필요한 것들은 따로 설치해야 한다.


*   transformers : 가장 핵심이다. GPT, BERT, LLaMA와 같은 사전 학습 모델을 불러오고 다루는 도구이다. Hugging Face에서 만든 라이브러리로, LoRA 실습의 주인공
*   matplotlib : 그래프를 그리는 도구이다. 학습 손실이 줄어드는 과정 등을 시각화할 때 쓴다.

*   rich : 터미널 출력을 예쁘게 만들어주는 도구이다. 진행 상황이나 결과를 보기 좋게 출력할 때 사용한다.
*   tqdm : 진행률 바를 만들어주는 도구이다. 학습이 얼마나 진행되었는지, [===> 60%] 이런 식으로 보여준다.





In [ ]:
# 필요 라이브러리 설치
!pip install -q transformers==4.57.6 matplotlib==3.10.8 rich==13.9.4 tqdm==4.67.3
# Colab 기본 설치 패키지는 보존: torch/torchvision/torchaudio/numpy/pandas


## GPU 확인

**현재 런타임의 GPU와 PyTorch CUDA 인식 상태를 확인한다.** Colab에서는 환경 설정과 패키지 설치 전후에 한 번 확인하면 된다.


Colab은 GPU를 자동으로 붙여주지 않는다. 설정에서 직접 GPU 런타임을 선택해야 하고, 선택을 안 했으면 CPU만 있는 상태이다. LoRA 학습은 GPU 없이는 사실상 불가능하므로, 시작 전에 먼저 확인하는 것이다.

In [ ]:
# GPU 확인
# nvidia-smi는 현재 런타임이 어떤 GPU를 받았는지 확인하는 가장 직접적인 방법이다.
import subprocess

try:
    subprocess.run(['nvidia-smi'], check=False)
except FileNotFoundError:
    print('nvidia-smi 명령을 찾을 수 없습니다. GPU 런타임인지 확인하세요.')

import torch

# 이후 학습 셀에서 사용할 torch 기준 CUDA 사용 가능 여부도 함께 확인한다.
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## 공통 import

노트북 전체에서 사용하는 외부 패키지와 저장소 공용 모듈을 한 번만 불러온다.


In [ ]:
# 공통 import
# Section 1의 toy LoRA 구현과 Section 2의 비용 분석에서 함께 쓰는 모듈을 한 곳에서 불러온다.

# 표준 라이브러리: 설정 객체, 경로 처리, base layer 복사에 사용한다.
import copy # 객체를 복사할 때 사용, 원본 모델을 건드리지 않고 복사본을 만들 때 사용한다.
import os # 운영체제 관련 기능, 환경변수 읽기 등을 담당
from dataclasses import dataclass # 설정값을 담는 클래스를 간편하게 만들 때 사용한다.
from pathlib import Path # 파일 경로를 다루는 도구

# Colab/서버 환경에서 matplotlib 설정 디렉터리 권한 문제를 피하기 위한 경로이다.
MPLCONFIGDIR = Path(os.environ.get("MPLCONFIGDIR", "/tmp/matplotlib"))
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))

# 딥러닝과 모델 구조 확인에 필요한 핵심 라이브러리이다.
import torch # 딥러닝 연산 전반을 담당
import transformers # GPT, BERT 같은 사전 학습 모델을 불러오는 Hugging Face 라이브러리
from torch import nn # 신경망 레이어를 만드는 도구, LoRA의 A,B 행렬도 nn으로 구현한다.
# 결과를 터미널에 보기 좋은 표 형태로 출력할 때 쓴다.
from rich.console import Console
from rich.table import Table

# utils/ 모듈은 노트북에서 반복되기 쉬운 장치 선택, 로깅, 리포트 생성을 담당한다.
from utils.device_utils import get_device_info, resolve_device # CPU/GPU중 어떤 것을 사용할지 판단하고 설정
from utils.logger_utils import configure_logger, get_logger # 학습 중 어떤 일이 일어나는지 기록하고 출력
# LoRA 비용 분석 관련 : LoRA 적용 시 파라미터가 몇 개인지, 메모리가 얼마나 드는지 계산하고 리포트를 만든다.
from utils.lora_cost import (
    build_lora_report,
    build_parameter_report,
    estimate_lora_trainable_params,
    collect_linear_shapes,
    collect_target_module_names,
    estimate_memory_profiles,
    format_bytes,
    load_model_from_config,
)
from utils.lora_toy import make_problem # 실습용 간단한 LoRA 문제를 만들어준다.
from utils.report_utils import build_fixed_width_table
from utils.training_utils import count_parameters
# 랭크별 파라미터 수 비교, 비용 비교 등을 그래프로 그려준다.
from utils.visualization import (
    show_lora_cost_comparison,
    show_lora_rank_summary,
)


# Section 1: LoRA Linear 직접 구현

Frozen `nn.Linear` 위에 low-rank A/B adapter를 더하는 `LoRALinear`를 구현한다. toy problem은 이미지 captioning으로 가기 전에 LoRA의 핵심 구조만 작게 떼어 내어 확인하는 선형 회귀 문제다. 여기서는 base layer를 고정한 상태에서 adapter만 학습해 full fine-tuning과 loss, trainable parameter를 비교한다.

## 실행 설정과 상수

LoRA 구조를 처음 확인하는 toy problem 조건을 정의한다.

- 실제 VLM 학습 전에 작은 `nn.Linear` 문제로 adapter가 무엇을 학습하는지 확인한다.
- rank 후보는 `(1, 2, 4, 8)`만 사용하고, Section 2의 reference rank는 이 중 가장 큰 값으로 맞춘다.
- `device=None`은 CUDA가 있으면 GPU, 없으면 CPU를 자동 선택한다.


아래 코드는 LoRA 실습에 필요한 모든 설정값을 한 곳에 모아두는 코드이다.

In [ ]:
# 실행 설정과 상수 : 실행 중 일어나는 일들을 기록하는 로거를 만든다.
LOGGER = get_logger("week2.section1.linear")

# @dataclass는 아래와 같은 설정 클래스를 간편하게 만들어주는 파이썬 문법이다
@dataclass(slots=True)
# 설정값들의 모음
class LoRALinearConfig:
    # Toy 문제는 Linear layer 하나만 다루므로 입출력 차원을 명시적으로 둔다.
    input_dim: int = 128 # 입력 차원
    output_dim: int = 128 # 출력 차원
    # target weight와 base weight의 차이가 완전히 작은 rank는 아니도록 만들어 LoRA rank 차이를 관찰한다.
    true_rank: int = 32
    # train/eval 입력 샘플 수이다. eval은 학습에 쓰지 않은 입력에서 일반화 정도를 보기 위해 분리한다.
    train_samples: int = 512 # 학습에 쓸 데이터 512개
    eval_samples: int = 512 # 성능 평가에 쓸 데이터 512개
    # 모든 방법이 같은 조건에서 비교되도록 학습 step과 learning rate를 공유한다.
    steps: int = 400 # 총 400번 업데이트
    learning_rate: float = 1e-2 # 학습률은 0.01로 설정
    # Section 1과 Section 2에서 같은 rank 후보를 사용해 parameter 증가량과 성능 변화를 함께 읽는다.
    ranks: tuple[int, ...] = (1, 2, 4, 8) # LoRA를 랭크 1,2,4,8로 학습해서 성능 차이를 비교한다. 랭크가 커질수록 파라미터는 늘고 성능은 어떻게 달라지는지 관찰하는 것
    seed: int = 42 # 재현성, 난수 씨앗값을 고정하기, 이 값이 같으면 누가 돌려도 같은 결과가 나오게 된다. 실험 재현성을 위해 반드시 필요
    # None이면 resolve_device가 CUDA 사용 가능 여부에 따라 자동으로 장치를 고른다.
    device: str | int | None = None # 장치 설정, None으로 두면 GPU가 있으면 자동으로 GPU, 없으면 CPU를 선택한다.

# 로그 형식, 난수 seed, 실행 장치를 먼저 고정해야 뒤 셀의 결과를 재현할 수 있다.
configure_logger()
config_linear = LoRALinearConfig()
transformers.set_seed(config_linear.seed)
device = resolve_device(config_linear.device)
device_info = get_device_info(str(device))
LOGGER.info("Using device: %s", device_info["device"])
if "device_name" in device_info:
    LOGGER.info("GPU: %s", device_info["device_name"])


## LoRALinear 계층 정의

고정된 base `nn.Linear` 위에 low-rank adapter를 더하는 최소 LoRA layer를 구현한다.

- base layer parameter는 freeze한다.
- `lora_a`, `lora_b`만 학습 가능하다.
- `alpha / rank` scaling으로 LoRA update 크기를 조절한다.

아래의 코드는 LoRA를 실제로 구현한 핵심 코드이다.

In [ ]:
# LoRALinear 계층 정의

# 파이썬에서 신경망 레이어를 만들 때에는 nn.Module을 상속받아서 만든다.
# LoRALinear는 Linear 레이어에 LoRA를 붙인 새로운 레이어이다.
class LoRALinear(nn.Module):
    # layer가 처음 만들어질 때 실행되는 부분
    def __init__(self, base_layer: nn.Linear, rank: int, alpha: float | None = None) -> None:
        super().__init__()
        if rank <= 0:
            raise ValueError("rank must be positive.")

        # LoRA는 원래 weight를 직접 학습하지 않고 frozen base layer로 보관한다.
        # deepcopy를 사용해 바깥에서 받은 base_layer와 이 모듈 내부 weight가 서로 영향을 주지 않게 한다.
        self.base_layer = copy.deepcopy(base_layer)
        for parameter in self.base_layer.parameters():
            parameter.requires_grad = False # 원본 가중치는 학습 중에 절대 바뀌지 않도록 얼려두기

        # alpha/rank는 LoRA 논문에서 쓰는 scaling이다. alpha를 생략하면 rank와 같게 두어 scaling=1이 된다.
        self.rank = rank
        self.alpha = float(rank if alpha is None else alpha)
        self.scaling = self.alpha / self.rank

        # W_delta = B @ A 형태의 저랭크 update를 두 Linear layer로 구현한다.
        # [Hint] lora_a: base_layer.in_features 차원에서 rank 차원으로 매핑하는 nn.Linear를 만드세요. (bias=False)
        # nn.Linear는 행렬의 곱을 의미한다.
        self.lora_a = nn.Linear(base_layer.in_features, rank, bias=False)
        # [Hint] lora_b: rank 차원에서 base_layer.out_features 차원으로 매핑하는 nn.Linear를 만드세요. (bias=False)
        self.lora_b = nn.Linear(rank, base_layer.out_features, bias=False)

        # A는 작은 난수, B는 0으로 초기화한다.
        # (빈칸을 올바르게 채워 넣었다면 아래 초기화 코드가 동작합니다.)
        # pytorch에서 함수 이름 뒤에 _가 붙으면 "제자리에서 직접 바꾼다. in-place"라는 뜻이다.
        # _가 없는 경우 -> 새로운 값을 반환, 원본은 그대로
        # _가 있는 경우 -> 원본을 직접 수정, 반환값 없음
        if self.lora_a is not None and self.lora_b is not None:
            nn.init.normal_(self.lora_a.weight, mean=0.0, std=0.02) # normal_ 정규분포(가우시간)로 초기화한다는 뜻
            nn.init.zeros_(self.lora_b.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # y = xW_base^T + scaling * xA^T B^T
        # [Hint] base_layer(x)의 결과에, x를 lora_a, lora_b 순서로 통과시키고 scaling을 곱한 값을 더하세요.
        return self.base_layer(x) + self.lora_b(self.lora_a(x))*self.scaling
        # 1단계: x (128차원) → lora_a → (4차원), 2단계: (4차원)     → lora_b → (128차원)


# --- 검증 코드 ---
try:
    _test_base = nn.Linear(16, 8, bias=True)
    _test_lora = LoRALinear(_test_base, rank=4)
    _test_input = torch.randn(2, 16)
    _test_output = _test_lora(_test_input)

    if _test_output is not None and _test_output.shape == (2, 8):
        print(f"성공! 입력: {tuple(_test_input.shape)} -> 출력: {tuple(_test_output.shape)}")
    else:
        print("실패! 기대되는 입출력 shape는 입력 (2, 16) -> 출력 (2, 8) 입니다. 또는 아직 코드를 완성하지 않았습니다.")
except Exception as e:
    print(f"실패! 코드를 완성해주세요. (오류 메시지: {e})")


## Linear와 LoRA 초기 상태 점검

`nn.Linear`의 parameter shape와 `requires_grad`를 확인하고, `lora_b`가 0으로 초기화되어 첫 forward 출력이 base layer와 같은지 점검한다.

아래의 코드는 일반 Linear와 LoRA Linear를 비교해서 초기 상태가 올바른지 점검하는 코드이다.

In [ ]:
# Linear와 LoRA 초기 상태 점검

# 먼저 일반 Linear layer의 weight/bias shape과 학습 대상 여부를 확인한다.
linear = nn.Linear(4, 3, bias=True).to(device) # nn.Linear는 숫자 변환기 : 숫자 4개를 넣으면 숫자 3개가 나오는 변환
x = torch.tensor([[1.0, 2.0, 3.0, 4.0]], device=device)
LOGGER.info("weight shape: %s", tuple(linear.weight.shape))
LOGGER.info("bias shape: %s", tuple(linear.bias.shape))
LOGGER.info("forward output: %s", linear(x).detach().cpu().tolist())
for name, parameter in linear.named_parameters():
    LOGGER.info("%s requires_grad=%s", name, parameter.requires_grad)

# 같은 base layer를 LoRA로 감싸면 base weight는 frozen이고 adapter만 학습 대상이 된다.
base_layer = nn.Linear(4, 3, bias=True).to(device) # 원본 Linear layer 만들기
lora_layer = LoRALinear(base_layer, rank=2).to(device)
probe = torch.randn(3, 4, device=device) # 샘플 3개, 각 샘플마다 숫자 4개

# [Hint] lora_b를 0으로 초기화했으므로, 학습 전 LoRA layer의 출력은 base layer의 출력과 같아야 합니다. 빈칸에 알맞은 코드를 작성하세요.
outputs_match = torch.allclose(base_layer(probe), lora_layer(probe), atol=1e-6)

# (빈칸을 올바르게 채워 넣었다면 아래 코드들이 정상 작동합니다.)
LOGGER.info("initial outputs match: %s", outputs_match)
for name, parameter in lora_layer.named_parameters():
    LOGGER.info("%s requires_grad=%s", name, parameter.requires_grad)


## Toy 학습 루프

Full fine-tuning baseline과 LoRA adapter가 공유하는 MSE 학습 루프다. 학습 가능한 parameter만 optimizer에 전달해 같은 loss와 learning rate로 비교한다.

# 실제 학습 vs Toy 학습

실제 LoRA 학습은 다음과 같다.

```
모델: LLaMA (70억 개 파라미터)
데이터: 수십만 개의 문장
목적: 챗봇, 요약, 번역 등
시간: 몇 시간 ~ 며칠
```

Toy 학습은 다음과 같다.


```
모델: nn.Linear 하나 (파라미터 수백 개)
데이터: 랜덤 숫자 512개
목적: LoRA가 올바르게 동작하는지 확인
시간: 몇 초
```

왜 Toy를 사용할까?
: LoRA 코드가 올바르게 작동하는지 확인하려면 실제 LLM을 쓰면 너무 오래걸린다. 그래서 구조는 똑같지만, 크기만 아주 작은 Toy 문제로 빠르게 테스트 하는 것


In [ ]:
# Toy 학습 루프

# Full FT와 LoRA가 같은 루프로 학습되도록 단순한 MSE regression 학습 함수를 둔다.
# train_model 함수는 모델을 학습시키고 결과를 반환하는 함수이다.
def train_model(
    model: nn.Module, # 학습시킬 모델(Full FT 또는 LoRA)
    *,# *은 이 뒤에 오는 인자들은 반드시 이름을 써서 전달해야 한다는 뜻 ex) x_train = x
    x_train: torch.Tensor, # 학습용 입력
    y_train: torch.Tensor, # 학습용 정답
    x_eval: torch.Tensor, # 평가용 입력
    y_eval: torch.Tensor, # 평가용 정답
    steps: int, # 몇 번 학습할지
    learning_rate: float, # 학습률
) -> dict[str, float]:
    # [Hint] Toy 문제는 target linear mapping을 맞추는 회귀 문제이므로 MSE 손실 함수를 사용합니다.
    loss_fn = nn.MSELoss()

    # [Hint] Full FT와 LoRA 모두에 대응하기 위해, 모델의 파라미터(p) 중 학습 가능한(requires_grad가 참인) 것만 골라 전달하세요.
    trainable_params = [p for p in model.parameters() if p.requires_grad] # model.paramters() , parameter.requires_grad
    # Full FT이면 모든 파라미터, LoRA면 lora_a, lora_b만 골라진다.

    # (빈칸을 올바르게 채워 넣었다면 아래 코드들이 정상 작동합니다.)
    optimizer = torch.optim.Adam(
        trainable_params,# trainable_params만 업데이트한다.
        lr=learning_rate,
    )
    log_every = max(1, steps // 4)

    # ------------------------------------------------------------------------------------------------------------------------
    # 학습 루프
    model.train()
    for step in range(steps):
        optimizer.zero_grad() # 이전 gradient 초기화

        # [Hint] 모델에 입력(x_train)을 통과시켜 예측값을 구하세요.
        preds = model(x_train)
        loss = loss_fn(preds, y_train)

        loss.backward()
        optimizer.step()
        if (step + 1) % log_every == 0 or step == 0:
            LOGGER.info("  step %d/%d train_loss=%.6f", step + 1, steps, loss.item())

    # 학습이 끝난 뒤 train/eval loss를 같은 기준으로 계산해 비교 표에 넣는다.
    # 학습 후 평가
    model.eval() # 평가 모드로 전환
    with torch.no_grad():
        train_loss = loss_fn(model(x_train), y_train).item()
        eval_loss = loss_fn(model(x_eval), y_eval).item()
    return {
        "train_loss": train_loss,
        "eval_loss": eval_loss,
    }


## Toy 문제 생성

Base weight와 target weight가 다른 작은 선형 회귀 문제를 만든다. Full FT는 전체 weight를 바꾸고, LoRA는 frozen base 위에 adapter update만 더해 같은 target을 맞춘다.

### 무슨 일을 하는 코드인가?

LoRA 실습을 위한 시험 문제를 만드는 코드이다. 구체적으로는 아래와 같은 것들을 만들어낸다.

```
base_weight   → 사전 학습된 모델의 가중치 (W₀)
target_weight → 우리가 맞춰야 할 정답 가중치
x_train       → 학습용 입력 데이터
y_train       → 학습용 정답 출력
x_eval        → 평가용 입력 데이터
y_eval        → 평가용 정답 출력
```

config_linear 안의 이런 값들을 사용
```
input_dim = 128
output_dim = 128
true_rank = 32
train_samples = 512
eval_samples = 512
seed = 42
```

반환값은 딕셔너리 형태이다.
```
toy_problem = {
    "x_train": ...,       # 학습 입력
    "y_train": ...,       # 학습 정답
    "x_eval": ...,        # 평가 입력
    "y_eval": ...,        # 평가 정답
    "base_weight": ...,   # W₀
    "target_weight": ..., # 정답 가중치
}
```






In [ ]:
# Toy 문제 생성

# make_problem은 base_weight와 target_weight, 그리고 target_weight로 만든 입출력 쌍을 생성한다.
# 학생은 여기서 LoRA가 target_weight 전체가 아니라 base_weight와의 차이를 저랭크 update로 맞춘다는 점을 본다.
toy_problem = make_problem(config_linear, device) # utils/lora_toy.py에 만들어진 함수이다.config_linear에 있는 설정값들을 보고 문제를 만든다.
LOGGER.info(
    "Toy tensors | x_train=%s y_train=%s base_weight=%s target_weight=%s",
    tuple(toy_problem["x_train"].shape), #(512,128)
    tuple(toy_problem["y_train"].shape), # (512,128)
    tuple(toy_problem["base_weight"].shape), #(128,128)
    tuple(toy_problem["target_weight"].shape), # (128,128)
)


## Full FT baseline 학습

같은 toy 문제에서 전체 `nn.Linear.weight`를 학습하는 기준선을 만든다.

In [ ]:
# Full FT baseline 학습

LOGGER.info("Running full fine-tuning baseline")

# Full FT baseline은 base_weight에서 시작하되, Linear weight 전체를 학습한다.
# Full FT이므로 rank 없음
full_ft_model = nn.Linear(config_linear.input_dim, config_linear.output_dim, bias=False).to(device)
with torch.no_grad():
    full_ft_model.weight.copy_(toy_problem["base_weight"]) # 시작 전에 가중치를 base_weight로 맞춰두는 것

# 같은 toy 데이터와 같은 학습 조건을 사용해 LoRA 결과와 직접 비교한다.
full_ft_metrics = train_model(
    full_ft_model,
    x_train=toy_problem["x_train"],
    y_train=toy_problem["y_train"],
    x_eval=toy_problem["x_eval"],
    y_eval=toy_problem["y_eval"],
    steps=config_linear.steps,
    learning_rate=config_linear.learning_rate,
)

# Full FT는 Linear weight 전체가 trainable parameter이다.
full_ft_total_params = count_parameters(full_ft_model)

try:
    # [Hint] Full FT는 전체 파라미터가 학습 대상입니다. 학습 가능한 파라미터 수만 세도록 알맞은 boolean 값을 입력하세요.
    rows = [
        {
            "model": "full_ft",
            "rank": "-",
            "trainable_params": count_parameters(full_ft_model, trainable_only=None),
            "trainable_ratio": count_parameters(full_ft_model, trainable_only=None) / full_ft_total_params,
            **full_ft_metrics,
        }
    ]
except Exception as e:
    print(f"실패! 코드를 완성해주세요. (오류 메시지: {e})")
    rows = []


## LoRA rank별 학습

Frozen base layer 위의 LoRA adapter만 학습한다. Rank가 커질수록 학습 가능한 parameter 수와 표현력이 함께 늘어난다.

아래의 코드는 rank를 1,2,4,8로 바꿔가며 LoRA를 학습하고 결과를 저장하는 코드이다.

```
rank=1 로 학습 → 결과 저장
rank=2 로 학습 → 결과 저장
rank=4 로 학습 → 결과 저장
rank=8 로 학습 → 결과 저장
        ↓
Full FT 결과와 한 표에 비교
```
rows 형식

```
rows = [
    {"model": "full_ft", "rank": "-",  "train_loss": ..., "eval_loss": ...},
    {"model": "lora",    "rank": "1",  "train_loss": ..., "eval_loss": ...},
    {"model": "lora",    "rank": "2",  "train_loss": ..., "eval_loss": ...},
    {"model": "lora",    "rank": "4",  "train_loss": ..., "eval_loss": ...},
    {"model": "lora",    "rank": "8",  "train_loss": ..., "eval_loss": ...},
]
```




In [ ]:
# LoRA rank별 학습

for rank in config_linear.ranks:
    LOGGER.info("Running LoRA with rank=%d", rank)

    # rank마다 같은 base_weight에서 새로 시작해야 rank 차이만 비교할 수 있다.
    base_layer = nn.Linear(config_linear.input_dim, config_linear.output_dim, bias=False).to(device)
    with torch.no_grad():
        base_layer.weight.copy_(toy_problem["base_weight"])

    # [Hint] 앞에서 구현한 LoRALinear를 이용해 base_layer를 감싸고, 현재 루프의 rank를 적용해 모델을 생성한 뒤 device로 보내세요.
    lora_model = LoRALinear(base_layer, rank=rank).to(device)

    # (빈칸을 올바르게 채워 넣었다면 아래 코드들이 정상 작동합니다.)
    lora_metrics = train_model(
        lora_model,
        x_train=toy_problem["x_train"],
        y_train=toy_problem["y_train"],
        x_eval=toy_problem["x_eval"],
        y_eval=toy_problem["y_eval"],
        steps=config_linear.steps,
        learning_rate=config_linear.learning_rate,
    )

    # total_params에는 frozen base와 LoRA adapter가 모두 포함되고, trainable_params는 adapter만 센다.
    total_params = count_parameters(lora_model) # 파라미터 수 집계
    trainable_params = count_parameters(lora_model, trainable_only=True) # 학습 대상인 파라미터 수 집계
    # 결과 저
    rows.append(
        {
            "model": "lora",
            "rank": str(rank),
            "trainable_params": trainable_params,
            "trainable_ratio": trainable_params / total_params,
            **lora_metrics,
        }
    )


## 결과 요약

Full FT와 rank별 LoRA 결과를 같은 table row 형식으로 정리한다.
= 지금까지 학습한 Full FT와 LoRA 결과를 보기 좋은 표로 출력하는 코드

> 두 가지의 숫자 포맷

```
f"{float(row['trainable_ratio']):.2%}"
# 0.0625 → "6.25%"  소수를 퍼센트로 변환

f"{float(row['train_loss']):.6f}"
# 0.000123456 → "0.000123"  소수점 6자리까지 표시
```



In [ ]:
# 결과 요약

# 로그 표는 숫자를 바로 비교하기 쉽도록 비율과 loss를 문자열로 포맷한다.
# rows에 저장된 결과들을 보기 좋은 형식으로 변환한다.
summary_rows = [
    {
        "model": row["model"],
        "rank": row["rank"],
        "trainable_params": row["trainable_params"],
        "trainable_ratio": f"{float(row['trainable_ratio']):.2%}",
        "train_loss": f"{float(row['train_loss']):.6f}",
        "eval_loss": f"{float(row['eval_loss']):.6f}",
    }
    for row in rows
]

# 표 출력, build_fixed_width_table은 결과를 표 형식으로 만들어준다.
LOGGER.info(
    "Summary\n%s",
    build_fixed_width_table(
        rows=summary_rows,
        columns=("model", "rank", "trainable_params", "trainable_ratio", "train_loss", "eval_loss"),
    ),
)


## rank별 LoRA 결과 시각화


`show_lora_rank_summary`로 full FT와 LoRA rank별 결과를 비교한다. 이 그래프는 작은 선형층에서도 rank 선택이 성능과 trainable parameter의 균형점이라는 것을 보여 주는 용도다.



> 그래프로 확인하려는 것

```
rank 키우면 → 파라미터 늘어남 (비용 증가)
rank 키우면 → eval loss 줄어듦 (성능 향상)
```





In [ ]:
# rank별 LoRA 결과 시각화

# trainable ratio와 eval loss를 함께 그려 rank를 키울 때의 비용-성능 변화를 확인한다.
# rows에 저장해둔 결과를 그래프로 시각화한다.
# 위에서 import할 때 불러온 함수이다. from utils.visualization import show_lora_rank_summary
show_lora_rank_summary(rows)


# Section 2: LoRA 비용 분석

SmolVLM 구조 기준으로 full fine-tuning과 reference rank LoRA의 trainable parameter 수, parameter-state memory 비용 차이를 계산한다.

## 실행 설정과 상수

SmolVLM에서 LoRA를 붙일 text projection module과 비용 비교에 사용할 reference rank를 정한다.

아래의 코드는 실제 LLM 모델에 LoRA를 적용할 때 비용을 분석하기 위한 설정값이다. Toy에서 벗어나 실제 모델을 분석한다.

> 실제 weight를 다운로드 하지 않는다의 의미

모델 전체를 받으면 수 GB가 필요하지만, 구조만 가져오면 파라미터 수 계산에 충분하기 때문이다.


```
SmolVLM
├── vision_model  → 이미지 처리 부분
└── text_model    → 텍스트 처리 부분
    └── layers    → 여기에 LoRA 붙임
```






> target_modules


Attention 부분(4개)
```
q_proj → Query  행렬  입력을 Query로 변환
k_proj → Key    행렬  입력을 Key로 변환
v_proj → Value  행렬  입력을 Value로 변환
o_proj → Output 행렬  Attention 결과를 출력으로 변환
```

FFN 부분(3개)
```
gate_proj → 어떤 정보를 통과시킬지 결정
up_proj   → 차원을 크게 늘림 (예: 512 → 2048)
down_proj → 차원을 다시 줄임 (예: 2048 → 512)
```





In [ ]:
# 실행 설정과 상수
LOGGER = get_logger("week2.section2.cost")

@dataclass(slots=True)
class AnalysisConfig:
    # 실제 weight를 다운로드하지 않고 config에서 모델 구조만 만들어 parameter 수를 계산한다.
    model_id: str = "HuggingFaceTB/SmolVLM-256M-Instruct" # hugging face에서 모델을 가져올 때 사용하는 주소

    # LoRA를 적용할 text decoder layer의 Linear projection 이름만 비용 분석 대상으로 삼는다.
    # SmolVLM은 이미지와 텍스트를 둘 다 처리하는 모델이다. 텍스트 처리 부분의 레이어만 LoRA의 분석 대상으로 삼음
    target_prefix: str = "model.text_model.layers."
    # 각 레이어 안에서 LoRA를 붙일 행렬들이다. 크게 두 그룹
    target_modules: tuple[str, ...] = (
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    )

    # Section 1에서 확인한 rank 후보 중 수업의 기준 비교값으로 사용할 rank이다.
    # section 1에서 rank를 1,2,4,8로 실험했는데, 그 중 rank=8을 기준값으로 삼아 실제 모델에서 비용을 계산
    reference_rank: int = 8


## 분석 환경 설정

LoRA 비용 분석 설정과 현재 device 정보를 준비한다.

In [ ]:
# 분석 환경 설정

config_cost = AnalysisConfig() # 위에서 정의한 AnalysisConfig 설정 객체 만들기
LOGGER.info("Step start: LoRA cost analysis")
LOGGER.info("Using model: %s", config_cost.model_id)

# 비용 계산 자체는 CPU에서도 가능하지만, 현재 실행 환경을 로그로 남겨 노트북 재현성을 높인다.
device_info = get_device_info()
LOGGER.info("Using device: %s", device_info["device"])


## 모델 구조와 full FT 기준 계산

Pretrained weight를 내려받지 않고 config로 SmolVLM architecture만 만든 뒤, full fine-tuning에서 학습되는 전체 parameter 수를 계산한다.
= 이 코드는 실제 모델의 구조만 가져와서 Full FT 기준 파라미터 수를 계산하는 코드이다.

In [ ]:
# 모델 구조와 full FT 기준 계산

LOGGER.info("Loading model architecture from config: %s", config_cost.model_id)

# from_pretrained weight 다운로드 없이 config 기반 empty model을 만들어 구조와 parameter 수만 확인한다.
model = load_model_from_config(config_cost.model_id) # 모델 구조 로드하기, load_model_from_config는 utils/lora_cost.py에 만들어진 함수
# hugging face에서 모델을 가져오되, 실제 가중치(weight)는 다운로드 하지 않고 구조만 가져온다.
full_total_params = count_parameters(model) # 파라미터 수 계산, 모델의 전체 파라미터 수 세기

# Full FT 기준에서는 모델의 모든 parameter가 학습 대상이라고 둔다.
# Full FT는 모든 파라미터가 학습 대상이므로 total_params와 trainable_params가 같다.
full_report = build_parameter_report(
    total_params=full_total_params,
    trainable_params=full_total_params,
)


## LoRA target module 선택

Text decoder의 projection layer 중 LoRA를 붙일 module을 찾고, 각 linear layer의 입출력 차원을 확인한다.
= 모델 전체에서 LoRA를 붙일 레이어들을 골라내는 코드이다.



> 두 가지 조건을 동시에 만족하는 레이어만 골라낸다.
```
조건 1: 이름이 "model.text_model.layers."로 시작
조건 2: 이름이 "q_proj", "k_proj", "v_proj" 등으로 끝남
```



> 결과 예시

```
target_module_names = [
    "model.text_model.layers.0.self_attn.q_proj",
    "model.text_model.layers.0.self_attn.k_proj",
    "model.text_model.layers.0.self_attn.v_proj",
    "model.text_model.layers.0.self_attn.o_proj",
    "model.text_model.layers.0.mlp.gate_proj",
    ...
    "model.text_model.layers.1.self_attn.q_proj",
    ...
]
```







In [ ]:
# LoRA target module 선택

# target_prefix 아래의 projection layer 중 suffix가 target_modules에 포함되는 Linear module만 고른다.
target_module_names = collect_target_module_names(
    model,
    prefix=config_cost.target_prefix, # "model.text_model.layers."
    suffixes=config_cost.target_modules, # ("q_proj", "k_proj", ...)
)

# 예외처리, 아무것도 찾지 못한 경우 오류
if not target_module_names:
    raise RuntimeError("No text target modules were found for LoRA analysis.")

# 각 Linear layer의 in/out feature shape이 있어야 rank별 LoRA parameter 수를 계산할 수 있다.
# 골라낸 layer 들의 입출력 차원을 수집한다. -> 이 정보가 있어야 rank별 파라미터 수를 계산할 수 있다.
linear_shapes = collect_linear_shapes(model, target_module_names)
LOGGER.info("Collected %s target modules for LoRA", len(target_module_names))


## Reference rank LoRA parameter 계산

선택한 target module의 shape를 사용해 reference rank adapter parameter 수를 계산한다.




> build_lora_report의 결과 형식

```
total_params:      257,000,000
trainable_params:  1,000,000
trainable_ratio:   0.39%        ← 전체 중 학습하는 비율
rank:              8
target_modules:    182개
```





In [ ]:
# Reference rank LoRA parameter 계산
# rank = 8로 LoRA 적용했을 때 파라미터가 몇 개 인지 계산하는 코드

try:
    # [Hint] LoRA 파라미터 수는 각 Linear layer의 shape과 rank에 의해 결정됩니다. 설정 객체(config_cost)에 정의된 reference_rank를 전달하세요.
    lora_trainable_params = estimate_lora_trainable_params(
        linear_shapes=linear_shapes,
        rank = config_cost.reference_rank, # AnalysisConfig에서 정의
    )

    # LoRA 모델은 frozen base weight를 유지하면서 adapter parameter를 추가로 가진다.
    # LoRA는 원본 모델을 건드리지 않고 A,B 행렬을 추가한다. 따라서 전체 파라미터는 원본 + 추가분
    lora_total_params = full_total_params + lora_trainable_params
    # 4가지를 받아서 리포트를 만든다.
    lora_report = build_lora_report(
        total_params=lora_total_params, # 전체 파라미터 수
        trainable_params=lora_trainable_params, # 학습 가능한 파라미터 수
        rank=config_cost.reference_rank, # rank = 8
        target_module_count=len(target_module_names), # 몇 개의 레이어에 붙엿는지
    )
except Exception as e:
    print(f"실패! 코드를 완성해주세요. (오류 메시지: {e})")
    lora_report = {
        "rank": config_cost.reference_rank,
        "trainable_params": 0,
        "trainable_ratio": 0
    }


## Parameter-state memory 계산

Full fine-tuning과 LoRA를 같은 mixed precision 기준으로 비교한다. 둘 다 forward에 필요한 model weight는 FP16으로 메모리에 올라간다고 가정한다.

- **Full FT:** 전체 parameter가 학습 대상이다. FP16 model weight에 더해, 전체 parameter에 대한 FP32 master weight, gradient, Adam optimizer state가 필요하다.
- **LoRA:** frozen base weight도 forward에 필요하므로 FP16 model weight로 메모리에 포함한다. 다만 base weight는 학습하지 않으므로 base weight에 대한 FP32 master weight, gradient, Adam state는 만들지 않는다.
- **LoRA adapter:** 학습되는 adapter parameter에 대해서만 FP16 weight, FP32 master weight, gradient, Adam optimizer state를 계산한다.
- **계산식:** `weights_and_master + gradients + optimizer_state`를 더해 total memory를 만든다.
- **제외 항목:** activation, dataloader memory, KV cache, sequence length 영향, memory fragmentation은 포함하지 않는다.


아래의 코드는 Full FT와 LoRA를 학습할 때 메모리가 얼마나 필요한지 계산하는 코드이다.


> 메모리가 왜 중요한가

학습 중에는 가중치만 저장하는 게 아니다.

```
Full FT 학습 시 필요한 메모리
─────────────────────────────
FP16 model weight   → 모델 가중치 (절반 정밀도)
FP32 master weight  → 정밀한 계산을 위한 복사본
gradient            → 역전파 결과
Adam state          → 학습률 조절용 (파라미터당 2개)
```
이걸 전부 합치면 모델 크기의 몇 배가 필요하다.

> Full FT vs LoRA 메모리 차이
```
Full FT
→ 256,000,000개 전부에 대해 위 4가지 필요

LoRA
→ FP16 weight: 256,000,000개 (원본 유지)
→ 나머지:      lora_trainable_params개만 필요
```




In [ ]:
# Parameter-state memory 계산

LOGGER.info("Estimating parameter-state memory profiles")

# 이 계산은 학습 중 parameter와 optimizer state가 차지하는 메모리만 비교한다.
# Full FT: 전체 parameter에 FP16 model weight, FP32 master weight, gradient, Adam state가 필요하다.
# LoRA: frozen base의 FP16 model weight는 유지하고, adapter에만 FP16/FP32/gradient/Adam state를 더한다.
try:
    # [Hint] LoRA 학습 시 실제로 업데이트되는 파라미터 수만 optimizer state와 gradient 메모리를 차지합니다. 위에서 생성한 lora_report 딕셔너리에서 학습 가능한 파라미터 수를 찾아 정수형(int)으로 입력하세요.
    cost_summary = estimate_memory_profiles(
        total_params=full_total_params,
        trainable_params=int(lora_report["trainable_params"]),
    )
except Exception as e:
    print(f"실패! 코드를 완성해주세요. (오류 메시지: {e})")
    cost_summary = {
        "full_finetuning": {"weight_bytes": 0, "gradient_bytes": 0, "optimizer_state_bytes": 0, "total_bytes": 0},
        "lora": {"weight_bytes": 0, "gradient_bytes": 0, "optimizer_state_bytes": 0, "total_bytes": 0}
    }

## 결과 요약

Full FT 기준과 reference rank LoRA 비용을 `rich.Table`로 비교한다. `weights+master`에는 forward에 필요한 FP16 model weight와 trainable parameter의 FP32 master weight가 포함된다.




> 숫자 포맷

```
f"{int(full_report['trainable_params']):,}"
# 256000000 → "256,000,000"  천 단위 콤마

format_bytes(cost_summary["full_finetuning"]["weight_bytes"])
# 3,070,000,000 → "2.86 GB"  바이트를 GB로 변환
```


> rich 라이브러리로 아래와 같은 표를 만든다.

```
Section 2 LoRA Parameter-State Memory
─────────────────────────────────────────────────────────────────────
method  rank  trainable_params  trainable_ratio  weights+master  gradients  optimizer  total_memory
─────────────────────────────────────────────────────────────────────
Full FT  -    256,000,000       100.00%          2.86 GB         ...        ...        ...
LoRA     8    1,000,000         0.39%            ...             ...        ...        ...
```


> 표에서 읽을 수 있는 것

```
trainable_params  → LoRA가 얼마나 적은 파라미터를 학습하는가
trainable_ratio   → 전체 대비 몇 %인가
weights+master    → 가중치 저장에 필요한 메모리
gradients         → 역전파에 필요한 메모리
optimizer         → Adam 상태에 필요한 메모리
total_memory      → 학습에 필요한 전체 메모리
```








In [ ]:
# 결과 요약
# Full FT와 LoRA의 메모리 사용량을 보기 좋은 표로 출력하는 코드이다.

# Full FT와 reference rank LoRA의 parameter-state memory를 사람이 읽기 쉬운 단위로 정리한다.
cost_rows = [
    {
        "method": "Full FT",
        "rank": "-",
        "trainable_params": f"{int(full_report['trainable_params']):,}",
        "trainable_ratio": f"{float(full_report['trainable_ratio']):.2%}",
        "weights+master": format_bytes(cost_summary["full_finetuning"]["weight_bytes"]),
        "gradients": format_bytes(cost_summary["full_finetuning"]["gradient_bytes"]),
        "optimizer": format_bytes(cost_summary["full_finetuning"]["optimizer_state_bytes"]),
        "total_memory": format_bytes(cost_summary["full_finetuning"]["total_bytes"]),
    },
    {
        "method": "LoRA",
        "rank": str(lora_report["rank"]),
        "trainable_params": f"{int(lora_report['trainable_params']):,}",
        "trainable_ratio": f"{float(lora_report['trainable_ratio']):.2%}",
        "weights+master": format_bytes(cost_summary["lora"]["weight_bytes"]),
        "gradients": format_bytes(cost_summary["lora"]["gradient_bytes"]),
        "optimizer": format_bytes(cost_summary["lora"]["optimizer_state_bytes"]),
        "total_memory": format_bytes(cost_summary["lora"]["total_bytes"]),
    },
]

# rich.Table을 사용해 terminal/Colab output에서 정렬된 표로 보여준다.
cost_columns = (
    "method",
    "rank",
    "trainable_params",
    "trainable_ratio",
    "weights+master",
    "gradients",
    "optimizer",
    "total_memory",
)

cost_table = Table(title="Section 2 LoRA Parameter-State Memory") # table title
for column in cost_columns:
    cost_table.add_column(column) # 열 추가
for row in cost_rows:
    cost_table.add_row(*(str(row[column]) for column in cost_columns)) # 행 추가
Console().print(cost_table) # 출력


## LoRA 비용 시각화

Full FT와 reference rank LoRA의 total parameter-state memory를 GB 축과 MB/GB label로 비교한다. LoRA bar에도 frozen base weight의 FP16 메모리가 포함되어 있다.


In [ ]:
# LoRA 비용 시각화

# 표에서 본 memory 항목을 막대그래프로 다시 보여주어 Full FT와 LoRA의 차이를 한눈에 비교한다.
# 아까 import할 때 불러온 함수이다. from utils.visualization import show_lora_cost_comparison
show_lora_cost_comparison(
    cost_summary["full_finetuning"], # Full FT 메모리 데이터
    cost_summary["lora"], # LoRA 메모리 데이
)
